# Banana Ripeness — DINOv3 vs ConvNeXt → KMeans

Same clean, bg-removed pool as `banana-clean.ipynb`; only the embedding changes.
DINOv3 is self-supervised (no ImageNet labels), so its features should keep colour / texture / damage cues that the label-trained ConvNeXt collapses into "banana".

**Requires:** `banana-clean.ipynb` steps 1–5 already run (reads `NOBG_DIR`).

1. Load the bg-removed pool + labels, filename prefix (source), original split (augmented or not).
2. Embed: ConvNeXt-Tiny barebone (baseline), DINOv3 ViT-S+/16 (CLS token, mean patch token).
3. KMeans (`k=4`) per feature variant → comparison table.
4. Inspect the best variant: confusion, source cross-tab, augmented vs. not.

In [1]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, confusion_matrix, normalized_mutual_info_score
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny
from torchvision.transforms import v2

In [2]:
# Config
WORK_DIR = Path("/tmp/bananafp")
DATASET_DIR = WORK_DIR / "Banana Ripeness Classification Dataset"  # raw, for original split lookup
NOBG_DIR = WORK_DIR / "clean_nobg"  # output of banana-clean.ipynb step 5
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

HUB_DIR = Path.home() / ".cache/torch/hub"
DINO_REPO = HUB_DIR / "facebookresearch_dinov3_main"
DINO_MODEL = "dinov3_vits16plus"
DINO_WEIGHTS = HUB_DIR / "checkpoints/dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth"
DINO_RES = 224  # multiple of patch size 16

N_CLUSTERS = 4  # overripe / ripe / rotten / unripe
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

## 1. Load the clean pool

In [3]:
assert NOBG_DIR.exists(), "run banana-clean.ipynb steps 1-5 first"

paths = sorted(p for p in NOBG_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
y = np.array([p.parent.name for p in paths])
prefix = np.array([re.match(r"musa-acuminata-([a-z]+)-", p.name).group(1) for p in paths])

# original split: train files are Roboflow-augmented, valid/test are not
split_by_name = {p.name: p.relative_to(DATASET_DIR).parts[0] for p in DATASET_DIR.rglob("*.jpg")}
orig_split = np.array([split_by_name[p.name] for p in paths])
augmented = orig_split == "train"

print(f"{len(paths)} images")
print(Counter(y))
print("augmented (from train):", augmented.sum(), "| not augmented (valid/test):", (~augmented).sum())

5199 images
Counter({np.str_('rotten'): 1676, np.str_('ripe'): 1571, np.str_('overripe'): 1098, np.str_('unripe'): 854})
augmented (from train): 3652 | not augmented (valid/test): 1547


## 2. Embeddings

In [4]:
@torch.no_grad()
def embed(paths, model_fn, transform, batch_size: int = 64) -> np.ndarray:
    vecs = []
    for i in range(0, len(paths), batch_size):
        batch = torch.stack([transform(Image.open(p).convert("RGB")) for p in paths[i : i + batch_size]])
        vecs.append(model_fn(batch.to(DEVICE)).float().cpu())
    return torch.cat(vecs).numpy()

In [5]:
# Baseline: ConvNeXt-Tiny barebone (conv stages + global avg pool, no head) -> (768,)
cnx_weights = ConvNeXt_Tiny_Weights.DEFAULT
_convnext = convnext_tiny(weights=cnx_weights)
convnext = torch.nn.Sequential(_convnext.features, _convnext.avgpool, torch.nn.Flatten(1)).eval().to(DEVICE)

X_convnext = embed(paths, convnext, cnx_weights.transforms())
print("convnext:", X_convnext.shape)

convnext: (5199, 768)


In [6]:
# DINOv3 ViT-S+/16 backbone from the local hub cache (no head)
dino = torch.hub.load(str(DINO_REPO), DINO_MODEL, source="local", weights=str(DINO_WEIGHTS))
dino = dino.eval().to(DEVICE)

# LVD-1689M weights -> standard ImageNet eval transform (per DINOv3 README)
dino_tf = v2.Compose([
    v2.ToImage(),
    v2.Resize((DINO_RES, DINO_RES), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

def dino_feats(x):
    out = dino.forward_features(x)
    cls = out["x_norm_clstoken"]  # (B, 384) global summary
    patch = out["x_norm_patchtokens"].mean(1)  # (B, 384) avg over patches: texture / spots
    return torch.cat([cls, patch], dim=1)

_dino = embed(paths, dino_feats, dino_tf)
X_dino_cls, X_dino_patch = _dino[:, :384], _dino[:, 384:]
print("dino cls:", X_dino_cls.shape, "| dino patch-mean:", X_dino_patch.shape)

dino cls: (5199, 384) | dino patch-mean: (5199, 384)


## 3. KMeans per variant

In [7]:
def l2n(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True).clip(min=1e-9)

def cluster_eval(X):
    """L2-normalise -> KMeans -> majority-vote label per cluster."""
    clusters = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=SEED).fit_predict(l2n(X))
    c2l = {}
    for c in range(N_CLUSTERS):
        vals, counts = np.unique(y[clusters == c], return_counts=True)
        c2l[c] = vals[int(np.argmax(counts))]
    y_pred = np.array([c2l[c] for c in clusters])
    metrics = {
        "acc": (y_pred == y).mean(),
        "acc_not_aug": (y_pred == y)[~augmented].mean(),
        "acc_aug": (y_pred == y)[augmented].mean(),
        "NMI": normalized_mutual_info_score(y, clusters),
        "ARI": adjusted_rand_score(y, clusters),
        "largest_cluster": np.bincount(clusters).max() / len(y),
        "labels_covered": len(set(c2l.values())),
    }
    return metrics, y_pred

variants = {
    "convnext": X_convnext,
    "dino_cls": X_dino_cls,
    "dino_patch": X_dino_patch,
    "dino_cls+patch": np.hstack([l2n(X_dino_cls), l2n(X_dino_patch)]),  # equal weight
}

results, preds = {}, {}
for name, X in variants.items():
    results[name], preds[name] = cluster_eval(X)
pd.DataFrame(results).T.round(3)

,acc,acc_not_aug,acc_aug,NMI,ARI,largest_cluster,labels_covered
convnext,0.591,0.582,0.594,0.280,0.173,0.622,4.0
dino_cls,0.661,0.661,0.660,0.454,0.365,0.392,4.0
dino_patch,0.491,0.502,0.487,0.151,0.139,0.273,3.0
dino_cls+patch,0.611,0.631,0.602,0.391,0.306,0.324,4.0


- `acc_not_aug` vs `acc_aug`: kept file came from valid/test (unaugmented) vs train (Roboflow-augmented).
- `largest_cluster`: share of the pool in the biggest cluster (catch-all check; balanced ≈ 0.25–0.32).
- `labels_covered` < 4 means two clusters mapped to the same label, so one class is never predicted.

## 4. Inspect the best variant

In [8]:
best = max(results, key=lambda k: results[k]["NMI"])
y_pred = preds[best]
classes = sorted(set(y))
print("best by NMI:", best)
pd.DataFrame(confusion_matrix(y, y_pred, labels=classes), index=classes, columns=classes).rename_axis(index="true", columns="pred")

best by NMI: dino_cls


pred,overripe,ripe,rotten,unripe
true,,,,
overripe,1078,0,0,20
ripe,404,1076,63,28
rotten,555,56,1062,3
unripe,0,595,41,218


In [9]:
# rows: folder class + filename prefix (source), cols: predicted
df = pd.DataFrame({"true": y, "prefix": prefix, "pred": y_pred})
pd.crosstab([df["true"], df["prefix"]], df["pred"], margins=True)

pred                  overripe  ripe  rotten  unripe   All
true     prefix                                           
overripe mold              368     0       0       4   372
         overripe          710     0       0      16   726
ripe     banana              5   458       9       1   473
         freshripe          27   487       2      15   531
         ripe              372   131      52      12   567
rotten   banana            364    13      22       0   399
         ripe                0     0     234       0   234
         rotten            191    26     562       3   782
         unripe              0    17     244       0   261
unripe   freshunripe         0   464       0     218   682
         unripe              0   131      41       0   172
All                       2037  1727    1166     269  5199